In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    hamming_loss,
    precision_score,
    recall_score,
    f1_score
)

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler, normalize
from scipy.sparse import hstack, csr_matrix
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve
from sklearn.preprocessing import MinMaxScaler, normalize
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold


In [20]:
#load the annotated dataset to train and test the model
from pathlib import Path
ROOT = Path.cwd().parent  # notebooks/ -> repo root--so tehd ata cna be laoded regardless of the file
df = pd.read_csv(ROOT / "annonated_training_test_set.csv")

label_cols = [
    "label_dependence",
    "label_attachment",
    "label_isolation_displacement",
    "label_crisis_response_frustration",
    "label_other_none"
]

# Keep rows with review text and fill missing review text with an empty string.
df = df.dropna(subset=["review"]).copy()
X = df["review"].astype(str)

# Yes = 1; blank or No = 0
y = (df[label_cols] == "Yes").astype(int)

print("Dataset shape:", df.shape)
print("Number of reviews:", len(X))
print("\nPositive examples per label:")
display(y.sum().rename("positive_count").to_frame())

Dataset shape: (300, 17)
Number of reviews: 300

Positive examples per label:


,positive_count
label_dependence,14
label_attachment,21
label_isolation_displacement,19
label_crisis_response_frustration,23
label_other_none,233


In [21]:
rarest = y.sum().idxmin()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y[rarest]
)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

print("\nPositive examples in each split:")
display(pd.DataFrame({
    "train_positive": y_train.sum(),
    "test_positive": y_test.sum()
}))

Training reviews: 210
Testing reviews: 90

Positive examples in each split:


,train_positive,test_positive
label_dependence,10,4
label_attachment,18,3
label_isolation_displacement,12,7
label_crisis_response_frustration,17,6
label_other_none,161,72


In [ ]:


# TF-IDF features are sparse and non-negative, so they can be used directly with MultinomialNB.
# use them for the training and testing of the model, and then combine them with the sentence transformer embeddings
tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)
T_train = tfidf.fit_transform(X_train)
T_test = tfidf.transform(X_test)

# sentence transformer embeddings are dense and can have negative values, so they need to be scaled and normalized before combining with TF-IDF features.
enc = SentenceTransformer("all-MiniLM-L6-v2")
E_train = enc.encode(X_train.tolist(), show_progress_bar=True)
E_test = enc.encode(X_test.tolist(), show_progress_bar=True)

# Naive Bayes needs non-negative features: scale to [0, 1] (fit on train only),
# then L2-normalize rows so the embedding block is on the same scale as TF-IDF
scaler = MinMaxScaler().fit(E_train)
E_train_s = normalize(scaler.transform(E_train))
E_test_s = normalize(np.clip(scaler.transform(E_test), 0, 1))

EMB_WEIGHT = 1.0   # try 0.5 / 2.0 to shift emphasis between embeddings and TF-IDF

X_train_tfidf = hstack([T_train, csr_matrix(EMB_WEIGHT * E_train_s)]).tocsr()
X_test_tfidf = hstack([T_test, csr_matrix(EMB_WEIGHT * E_test_s)]).tocsr()

print("TF-IDF features:", T_train.shape[1])
print("Embedding features:", E_train.shape[1])
print("Combined training matrix:", X_train_tfidf.shape)
print("Combined testing matrix:", X_test_tfidf.shape)

Batches: 100%|██████████| 3/3 [00:01<00:00,  2.98it/s]


TF-IDF features: 1348
Embedding features: 384
Combined training matrix: (210, 1732)
Combined testing matrix: (90, 1732)


In [23]:
from sklearn.naive_bayes import ComplementNB
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import precision_recall_curve, average_precision_score

ALPHA_GRID = [0.01, 0.1, 0.3, 0.5, 1.0, 3.0]

models = {}
thresholds = {}
best_alphas = {}
alpha_table = {}   # AP per alpha, per label, for inspection

y_pred = pd.DataFrame(index=y_test.index, columns=label_cols, dtype=int)
y_probability = pd.DataFrame(index=y_test.index, columns=label_cols, dtype=float)

for label in label_cols:
    # stratified k-fold cross-validation to tune the threshold for each label-- so that the model can be more sensitive to the positive class, which is often the minority class in imbalanced datasets
    ##cross_val predict is used to get the predicted probabilities for each sample in the training set, which are then used to compute the precision-recall curve and find the optimal threshold that maximizes the F1 score
    ## complementNB is a variant of the Multinomial Naive Bayes algorithm that is designed to handle imbalanced datasets better by using a different smoothing technique. It is particularly useful when the positive class is rare, as it can help improve the model's performance on the minority class.
    ## alpha is the smoothing strength. Instead of fixing it, we sweep ALPHA_GRID with CV on the training set only and keep the alpha with the best average precision for each label. Small alpha trusts rare words more; large alpha flattens the probabilities toward uniform. The test set is never used to pick alpha.
    n_pos = int(y_train[label].sum())
    n_splits = max(2, min(5, n_pos))
    cv = StratifiedKFold(n_splits, shuffle=True, random_state=42)

    # 1. Sweep alpha with CV on the training set only
    oofs, aps = {}, {}
    for a in ALPHA_GRID:
        oofs[a] = cross_val_predict(
            ComplementNB(alpha=a), X_train_tfidf, y_train[label],
            cv=cv, method="predict_proba"
        )[:, 1]
        aps[a] = average_precision_score(y_train[label], oofs[a])

    alpha_table[label] = aps
    best_a = max(aps, key=aps.get)
    best_alphas[label] = best_a

    # 2. Tune the threshold using the best alpha's out-of-fold probabilities
    p, r, t = precision_recall_curve(y_train[label], oofs[best_a])
    f1 = 2 * p * r / (p + r + 1e-9)
    thresholds[label] = t[np.argmax(f1[:-1])]

    # 3. Fit on the full training set with the best alpha, apply the tuned threshold to the test set
    model = ComplementNB(alpha=best_a)
    model.fit(X_train_tfidf, y_train[label])
    models[label] = model

    proba = model.predict_proba(X_test_tfidf)[:, 1]
    y_probability[label] = proba
    y_pred[label] = (proba >= thresholds[label]).astype(int)

print("Trained", len(models), "Complement Naive Bayes models.")
print("Best alpha per label:", best_alphas)
print("Thresholds:", {k: round(v, 3) for k, v in thresholds.items()})

print("\nTraining-CV average precision by alpha:")
display(pd.DataFrame(alpha_table).T.round(3).rename_axis("label"))

print("\nPredicted positives:\n", y_pred.sum())
print("\nActual positives:\n", y_test.sum())

Trained 5 Complement Naive Bayes models.
Best alpha per label: {'label_dependence': 0.01, 'label_attachment': 0.01, 'label_isolation_displacement': 0.01, 'label_crisis_response_frustration': 0.1, 'label_other_none': 0.3}
Thresholds: {'label_dependence': np.float64(0.676), 'label_attachment': np.float64(0.245), 'label_isolation_displacement': np.float64(0.147), 'label_crisis_response_frustration': np.float64(0.608), 'label_other_none': np.float64(0.637)}

Training-CV average precision by alpha:


,0.01,0.10,0.30,0.50,1.00,3.00
label,,,,,,
label_dependence,0.118,0.053,0.045,0.045,0.045,0.046
label_attachment,0.378,0.340,0.268,0.247,0.236,0.246
label_isolation_displacement,0.117,0.086,0.069,0.066,0.060,0.057
label_crisis_response_frustration,0.600,0.659,0.601,0.580,0.533,0.497
label_other_none,0.862,0.896,0.898,0.895,0.893,0.885



Predicted positives:
 label_dependence                      0
label_attachment                     12
label_isolation_displacement         11
label_crisis_response_frustration     2
label_other_none                     87
dtype: int64

Actual positives:
 label_dependence                      4
label_attachment                      3
label_isolation_displacement          7
label_crisis_response_frustration     6
label_other_none                     72
dtype: int64


In [24]:
## get the evaluation metrics for each label and store them in a DataFrame for analysis

## go through each label and print the classification report for each label, which includes precision, recall, f1-score, and support for each class (0 and 1). Also, calculate overall metrics such as subset accuracy, hamming loss, micro and macro precision, recall, and f1-score for the multi-label classification problem.
## precision: The ratio of correctly predicted positive observations to the total predicted positive observations. It indicates how many of the predicted positive cases were actually positive.
## recall: The ratio of correctly predicted positive observations to the all observations in actual class. It indicates how many of the actual positive cases were correctly predicted.
## f1-score: The weighted average of Precision and Recall. It is a better measure than accuracy for imbalanced datasets.    
## hamming loss: The fraction of labels that are incorrectly predicted. It is calculated as the number of incorrect labels divided by the total number of labels.
## micro precision, recall, and f1-score: These metrics are calculated by aggregating the contributions of all classes to compute the average metric. They are useful when you want to evaluate the overall performance of the model across all labels.
## macro precision, recall, and f1-score: These metrics are calculated by taking the average of the metrics for each class. They treat all classes equally, regardless of their support (number of true instances for each class). They are useful when you want to evaluate the model's performance on each label independently.
for label in label_cols:
    print("\n" + "=" * 75)
    print(label)
    print("=" * 75)
    print(classification_report(
        y_test[label],
        y_pred[label],
        labels=[0, 1],
        target_names=["No", "Yes"],
        zero_division=0
    ))

    overall_metrics = pd.DataFrame({
    "metric": [
        "Subset accuracy (exact match)",
        "Hamming loss",
        "Micro precision",
        "Micro recall",
        "Micro F1",
        "Macro precision",
        "Macro recall",
        "Macro F1"
    ],
    "score": [
        accuracy_score(y_test, y_pred),
        hamming_loss(y_test, y_pred),
        precision_score(y_test, y_pred, average="micro", zero_division=0),
        recall_score(y_test, y_pred, average="micro", zero_division=0),
        f1_score(y_test, y_pred, average="micro", zero_division=0),
        precision_score(y_test, y_pred, average="macro", zero_division=0),
        recall_score(y_test, y_pred, average="macro", zero_division=0),
        f1_score(y_test, y_pred, average="macro", zero_division=0)
    ]
})

display(overall_metrics)


label_dependence
              precision    recall  f1-score   support

          No       0.96      1.00      0.98        86
         Yes       0.00      0.00      0.00         4

    accuracy                           0.96        90
   macro avg       0.48      0.50      0.49        90
weighted avg       0.91      0.96      0.93        90


label_attachment
              precision    recall  f1-score   support

          No       0.96      0.86      0.91        87
         Yes       0.00      0.00      0.00         3

    accuracy                           0.83        90
   macro avg       0.48      0.43      0.45        90
weighted avg       0.93      0.83      0.88        90


label_isolation_displacement
              precision    recall  f1-score   support

          No       0.91      0.87      0.89        83
         Yes       0.00      0.00      0.00         7

    accuracy                           0.80        90
   macro avg       0.46      0.43      0.44        90
weighted

,metric,score
0,Subset accuracy (exact match),0.611111
1,Hamming loss,0.128889
2,Micro precision,0.651786
3,Micro recall,0.793478
4,Micro F1,0.715686
5,Macro precision,0.265517
6,Macro recall,0.233333
7,Macro F1,0.231132


In [ ]:
## check if embeddings, tf-idf, or a combonation of both works better for the multi-label classification

risk_labels = label_cols[:4]
X_arr = X.reset_index(drop=True)
Y = y[risk_labels].reset_index(drop=True)

E = enc.encode(X_arr.tolist(), show_progress_bar=True)   # enc from the earlier cell
EMB_WEIGHT = 3.0

mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = pd.DataFrame(0.0, index=Y.index, columns=risk_labels)

for tr, te in mskf.split(E, Y):
    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    T_tr = tfidf.fit_transform(X_arr.iloc[tr])
    T_te = tfidf.transform(X_arr.iloc[te])

    sc = MinMaxScaler().fit(E[tr])
    E_tr = normalize(sc.transform(E[tr]))
    E_te = normalize(np.clip(sc.transform(E[te]), 0, 1))

    F_tr = hstack([T_tr, csr_matrix(EMB_WEIGHT * E_tr)]).tocsr()
    F_te = hstack([T_te, csr_matrix(EMB_WEIGHT * E_te)]).tocsr()

    for label in risk_labels:
        nb = ComplementNB(alpha=0.5).fit(F_tr, Y.iloc[tr][label])
        oof.iloc[te, oof.columns.get_loc(label)] = nb.predict_proba(F_te)[:, 1]

print(f"{'label':38s} {'base rate':>9s} {'AP':>6s} {'best F1':>8s}")
for label in risk_labels:
    p, r, t = precision_recall_curve(Y[label], oof[label])
    best_f1 = (2 * p * r / (p + r + 1e-9)).max()
    print(f"{label:38s} {Y[label].mean():9.3f} "
          f"{average_precision_score(Y[label], oof[label]):6.3f} {best_f1:8.3f}")

Batches: 100%|██████████| 10/10 [00:03<00:00,  3.05it/s]


label                                  base rate     AP  best F1
label_dependence                           0.047  0.039    0.107
label_attachment                           0.070  0.206    0.286
label_isolation_displacement               0.063  0.088    0.182
label_crisis_response_frustration          0.077  0.510    0.564


In [ ]:


oof_lr = pd.DataFrame(0.0, index=Y.index, columns=risk_labels)
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for tr, te in mskf.split(E, Y):
    for label in risk_labels:
        lr = LogisticRegression(class_weight="balanced", max_iter=2000).fit(E[tr], Y.iloc[tr][label])
        oof_lr.iloc[te, oof_lr.columns.get_loc(label)] = lr.predict_proba(E[te])[:, 1]

for l in risk_labels:
    print(f"{l:38s} base {Y[l].mean():.3f}  LR AP {average_precision_score(Y[l], oof_lr[l]):.3f}")

label_dependence                       base 0.047  LR AP 0.120
label_attachment                       base 0.070  LR AP 0.279
label_isolation_displacement           base 0.063  LR AP 0.129
label_crisis_response_frustration      base 0.077  LR AP 0.565
